# ?lections l?gislatives marocaines : participation et repr?sentation

**Question directrice :** comment la participation moyenne des circonscriptions locales ?volue-t-elle entre les scrutins (2011, 2016, 2021), et comment la part des ?lues varie-t-elle entre les r?gions en 2021 ?

Les taux sont stock?s entre 0 et 1. Consultez le dictionnaire des classeurs pour interpr?ter les variables et gardez en t?te les limites d?crites dans ce notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == 'Script':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'Script'))
from prepare_data import load_year, FILES

sns.set_theme(style='whitegrid')
df = pd.concat([load_year(year, filename) for year, filename in FILES.items()], ignore_index=True, sort=False)
df.head()

## 1. Comprendre et vérifier le dataset
Chaque ligne correspond à une unité enregistrée dans le classeur. Vérifiez le type de liste et les colonnes de taux avant toute agrégation.

In [ ]:
print('Dimensions :', df.shape)
print('Années :', sorted(df['annee'].unique()))
display(df.dtypes)
display(df.isna().sum().sort_values(ascending=False).head(15))
display(df['typeListe'].value_counts(dropna=False))

## 2. Nettoyage et choix d?analyse
Les valeurs manquantes ne sont pas remplac?es par z?ro. Les scrutins n?incluent pas tous les m?mes types de listes : on garde les listes locales, pr?sentes dans les trois ann?es.

In [ ]:
required = ['annee', 'region', 'typeListe', 'txParticipation', 'nSieges', 'repPctFemmes']
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f'Colonnes attendues absentes : {missing}')
analysis = df[df['typeListe'].astype(str).str.lower() == 'locale'].copy()
analysis = analysis.dropna(subset=['region', 'txParticipation'])
for column in ['txParticipation', 'nSieges', 'repPctFemmes']:
    analysis[column] = pd.to_numeric(analysis[column], errors='coerce')
display(analysis[['txParticipation', 'nSieges', 'repPctFemmes']].describe())

## 3. Participation moyenne par ann?e
`nInscrits` est manquant en 2016 et 2021. On compare donc la moyenne des taux des circonscriptions locales; ce n?est pas le taux national pond?r? par les inscrits.

In [ ]:
participation = analysis.groupby('annee', as_index=False).agg(
    taux_moyen_circonscriptions=('txParticipation', 'mean'),
    nombre_circonscriptions=('txParticipation', 'count')
)
display(participation)
ax = sns.barplot(data=participation, x='annee', y='taux_moyen_circonscriptions', color='#2878B5')
ax.set(title='Participation moyenne des circonscriptions locales', xlabel='Ann?e', ylabel='Taux moyen (0?1)')
plt.tight_layout()
plt.show()

## 4. Part des ?lues par r?gion en 2021
Le dictionnaire d?finit `repPctFemmes` comme le pourcentage d??lues. Cette variable n?est fournie que pour 2021; l?agr?gation r?gionale est pond?r?e par le nombre de si?ges locaux.

In [ ]:
women = analysis.dropna(subset=['repPctFemmes', 'nSieges'])
women = women[(women['nSieges'] > 0) & (women['annee'] == 2021)].copy()
women['contribution_ponderee'] = women['repPctFemmes'] * women['nSieges']
regional = women.groupby('region', as_index=False).agg(
    contribution_ponderee=('contribution_ponderee', 'sum'),
    sieges=('nSieges', 'sum')
)
regional['representation_ponderee'] = regional['contribution_ponderee'] / regional['sieges']
display(regional)
plt.figure(figsize=(12, 6))
sns.barplot(data=regional, x='region', y='representation_ponderee', color='#D17A22')
plt.xticks(rotation=55, ha='right')
plt.title('Part des ?lues par r?gion ? 2021')
plt.xlabel('R?gion'); plt.ylabel('Part pond?r?e (0?1)')
plt.tight_layout(); plt.show()

## 5. ? compl?ter pour votre pr?sentation
- Quel r?sultat r?pond le mieux ? votre question ? Donnez une valeur pr?cise et son unit?.
- La participation moyenne des circonscriptions augmente-t-elle ou baisse-t-elle entre les scrutins ?
- Quelles r?gions ont les parts d??lues les plus ?lev?es en 2021 ?
- Quelles limites viennent des valeurs manquantes et des changements de fronti?res ou de d?coupage ?lectoral ?
- Quelle donn?e r?elle compl?mentaire permettrait de v?rifier ou d?approfondir votre interpr?tation ?